# Sentinel-2 Data Availability Comparison

Pick **one region of interest** by drawing it on the map, then this notebook compares how many Sentinel-2 scenes each free STAC API lists for that region, over the **entire archive**.

- **2 missions**: Sentinel-2 **L2A** and Sentinel-2 **L1C**
- **4 APIs**:
  - **element84** (Earth Search) - `https://earth-search.aws.element84.com/v1/`
  - **CDSE** (Copernicus Data Space Ecosystem, ESA authoritative) - `https://stac.dataspace.copernicus.eu/v1`
  - **planetary_computer** (Microsoft) - `https://planetarycomputer.microsoft.com/api/stac/v1`
  - **terrabyte** (DLR/LRZ) - `https://stac.terrabyte.lrz.de/public/api/`

Metric: number of **unique acquisition days** (solar days) returned for your ROI's bounding box, with no date filter and no cloud filter (raw archive availability). Raw item counts are also shown for reference.

> Notes:
> - planetary_computer hosts **L2A only**, so its L1C cells are reported as N/A.
> - Catalog *search* is anonymous on all four APIs; this notebook measures catalog listing only, not whether the pixels are anonymously downloadable.
> - A large ROI over the full archive means a lot of paging and can take several minutes.

## 1. Draw your study area

Use the draw tools at the **top-left** of the map to draw a **rectangle or polygon** over your area. A rectangle is recommended. You can pan/zoom anywhere in the world first. Then run the next cell to capture the bounding box.

In [ ]:
import leafmap

m = leafmap.Map(center=[48.1374, 11.5755], zoom=4, height="600px")
m

## 2. Capture the bounding box

Run this after drawing. Re-draw and re-run to change the ROI.

In [ ]:
def roi_to_bbox(m):
    """Return [minx, miny, maxx, maxy] from the last polygon/rectangle drawn on a leafmap Map.
    No GeoJSON file is written - we only read the in-memory geometry."""
    roi = m.user_roi
    if roi is None:
        raise ValueError("No ROI found. Draw a rectangle/polygon on the map above first, then re-run this cell.")
    geom = roi.get("geometry", roi)

    def _coords(c):
        if c and isinstance(c[0], (int, float)):
            yield c
        else:
            for x in c:
                yield from _coords(x)

    pts = list(_coords(geom["coordinates"]))
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    return [min(xs), min(ys), max(xs), max(ys)]


BBOX = roi_to_bbox(m)
print("ROI bbox [minx, miny, maxx, maxy]:", BBOX)

In [ ]:
import time
import pandas as pd
from pystac_client import Client

pd.set_option("display.max_rows", 200)

# mission -> api -> (url, collection).  None means the api does not offer that mission.
MISSIONS = {
    "S2_L2A": {
        "element84":          ("https://earth-search.aws.element84.com/v1/", "sentinel-2-l2a"),
        "cdse":               ("https://stac.dataspace.copernicus.eu/v1", "sentinel-2-l2a"),
        "planetary_computer": ("https://planetarycomputer.microsoft.com/api/stac/v1", "sentinel-2-l2a"),
        "terrabyte":          ("https://stac.terrabyte.lrz.de/public/api/", "sentinel-2-c1-l2a"),
    },
    "S2_L1C": {
        "element84":          ("https://earth-search.aws.element84.com/v1/", "sentinel-2-l1c"),
        "cdse":               ("https://stac.dataspace.copernicus.eu/v1", "sentinel-2-l1c"),
        "planetary_computer": None,  # PC has no L1C collection
        "terrabyte":          ("https://stac.terrabyte.lrz.de/public/api/", "sentinel-2-c1-l1c"),
    },
}
APIS = list(dict.fromkeys(a for mp in MISSIONS.values() for a in mp))

_CLIENTS = {}

def _client(url):
    if url not in _CLIENTS:
        _CLIENTS[url] = Client.open(url)
    return _CLIENTS[url]


def query_days(url, collection, bbox, limit=100, retries=3):
    """Entire-archive search (no datetime). Returns (raw_item_count, set_of_unique_days).

    Uses the STAC 'fields' extension to fetch only properties.datetime: CDSE caps limit at
    100 for sentinel-2-l2a without it and element84 500s on heavy full-archive responses,
    and lighter payloads are much faster. Iterates raw dicts via items_as_dicts() because a
    fields-trimmed response drops the top-level 'type' field that pystac Item parsing needs.
    Retries transient API errors (element84 occasionally returns a 500 on big queries).
    """
    client = _client(url)
    last = None
    for attempt in range(retries):
        try:
            search = client.search(collections=[collection], bbox=bbox, limit=limit,
                                   fields={"include": ["properties.datetime"]})
            days, raw = set(), 0
            for d in search.items_as_dicts():
                raw += 1
                dt = d.get("properties", {}).get("datetime")
                if dt:
                    days.add(dt[:10])
            return raw, days
        except Exception as e:
            last = e
            time.sleep(2 * (attempt + 1))
    raise last


RESULTS = {}  # mission -> {api: {"raw": int|None, "days": set|None}}


def evaluate(mission):
    """Query every api for one mission over the drawn ROI, store in RESULTS, return a DataFrame."""
    rows, store = [], {}
    for api, cfg in MISSIONS[mission].items():
        if cfg is None:
            store[api] = {"raw": None, "days": None}
            rows.append({"api": api, "raw_items": None, "unique_days": None, "collection": "- (not offered)"})
            continue
        url, coll = cfg
        try:
            raw, days = query_days(url, coll, BBOX)
            store[api] = {"raw": raw, "days": days}
            rows.append({"api": api, "raw_items": raw, "unique_days": len(days), "collection": coll})
        except Exception as e:
            store[api] = {"raw": None, "days": None}
            rows.append({"api": api, "raw_items": None, "unique_days": None, "collection": f"ERROR {type(e).__name__}: {e}"})
    RESULTS[mission] = store
    print(f"{mission}  |  bbox={BBOX}")
    return pd.DataFrame(rows).set_index("api")

# Sentinel-2 L2A

In [ ]:
evaluate("S2_L2A")

# Sentinel-2 L1C

In [ ]:
evaluate("S2_L1C")

# Report 1 - Available Scenes

Unique acquisition days per API for each mission, over your ROI. Run both `evaluate` cells above first.

In [ ]:
def available_table():
    data = {api: {} for api in APIS}
    for mission, store in RESULTS.items():
        for api in APIS:
            days = store.get(api, {}).get("days")
            data[api][mission] = (len(days) if days is not None else None)
    df = pd.DataFrame(data).T.reindex(APIS)
    return df[[m for m in MISSIONS if m in df.columns]]

available_table()

# Report 2 - Missing Scenes vs. Best API

For each mission, the API with the most unique days is the reference (`best_api`). `missing_vs_best` is how many of those reference days each API does **not** have. `0` = nothing missing relative to the best; `N/A` (None) = mission not offered by that API.

In [ ]:
def missing_table():
    records = []
    for mission, store in RESULTS.items():
        avail = {api: s["days"] for api, s in store.items() if s.get("days") is not None}
        if not avail:
            continue
        best_api = max(avail, key=lambda a: len(avail[a]))
        best_days = avail[best_api]
        for api in APIS:
            days = store.get(api, {}).get("days")
            records.append({
                "mission": mission,
                "api": api,
                "unique_days": (len(days) if days is not None else None),
                "missing_vs_best": (len(best_days - days) if days is not None else None),
                "best_api": best_api,
                "best_days": len(best_days),
            })
    return (pd.DataFrame(records)
            .set_index(["mission", "api"])
            .sort_index())

missing_table()

# Report 3 - Exact Missing Scenes

For each mission, lists the **actual acquisition dates** that each API is missing relative to the most complete API (`best_api` from Report 2). Printed per API, and also returned as a DataFrame whose `missing_dates` column holds the full date list.

In [ ]:
def missing_dates_report():
    rows = []
    for mission, store in RESULTS.items():
        avail = {api: s["days"] for api, s in store.items() if s.get("days") is not None}
        if not avail:
            continue
        best_api = max(avail, key=lambda a: len(avail[a]))
        best_days = avail[best_api]
        print()
        print(f"=== {mission} | reference (most complete): {best_api} ({len(best_days)} days) ===")
        for api in APIS:
            days = store.get(api, {}).get("days")
            if days is None:
                print(f"  {api}: N/A (mission not offered)")
                continue
            miss = sorted(best_days - days)
            print(f"  {api}: missing {len(miss)} date(s)")
            if miss:
                print("    " + ", ".join(miss))
            rows.append({"mission": mission, "api": api,
                         "missing_count": len(miss), "missing_dates": miss})
    return pd.DataFrame(rows).set_index(["mission", "api"])

missing_dates_report()